# Scraping product category
## Category level 1

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from lxml import etree
import time
from urllib.parse import urlparse, parse_qs

In [2]:
def get_category_lv1_title(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'th-TH,th;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.naiin.com/',
    }

    parsed_url = urlparse(url)
    params = parse_qs(parsed_url.query)
    p_id = params.get('product_type_id', [''])[0]
    cat1_code = params.get('category_1_code', [''])[0]

    # เตรียมข้อมูลพื้นฐานรอไว้ก่อน
    data = {
        'product_type_id': p_id,
        'category_1_code': cat1_code,
        'category_1_title': "-", # Default เป็น "-"
        'status_code': None,
        'url': url
    }

    try:
        response = session.get(url, headers=headers, timeout=15)
        data['status_code'] = response.status_code

        # ตรวจสอบว่าถ้าเป็น 200 OK ถึงจะทำการ Parse หา Title
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            h1_tag = soup.find('h1', class_='tw-text-center')
            if h1_tag:
                data['category_1_title'] = h1_tag.get_text(strip=True)
        else:
            print(f"Skipping Title: Received Status {response.status_code} for {url}")

    except Exception as e:
        print(f"Network/Request Error at {url}: {e}")
        data['status_code'] = "Error"

    return data

def get_product_level_1():
    sitemap_url = "https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_CategoryLv1.xml"
    print("Fetching Category Level 1 Sitemap...")

    try:
        response = requests.get(sitemap_url)
        root = etree.fromstring(response.content)
        namespaces = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        all_urls = root.xpath('//ns:loc/text()', namespaces=namespaces)
    except Exception as e:
        print(f"Failed to read sitemap: {e}")
        return

    target_urls = [u for u in all_urls]
    print(f"Total target Level 1 URLs: {len(target_urls)}")

    results = []

    with requests.Session() as session:
        for i, url in enumerate(target_urls):
            print(f"[{i+1}/{len(target_urls)}] Scraping: {url}")

            data = get_category_lv1_title(url, session)
            results.append(data)

            time.sleep(1) 

    df = pd.DataFrame(results)
    return df

In [3]:
cat_lv1_df = get_product_level_1()

Fetching Category Level 1 Sitemap...
Total target Level 1 URLs: 55
[1/55] Scraping: https://www.naiin.com/category?category_1_code=1&product_type_id=1
[2/55] Scraping: https://www.naiin.com/category?category_1_code=2&product_type_id=1
[3/55] Scraping: https://www.naiin.com/category?category_1_code=3&product_type_id=1
[4/55] Scraping: https://www.naiin.com/category?category_1_code=4&product_type_id=1
[5/55] Scraping: https://www.naiin.com/category?category_1_code=5&product_type_id=1
[6/55] Scraping: https://www.naiin.com/category?category_1_code=8&product_type_id=1
[7/55] Scraping: https://www.naiin.com/category?category_1_code=9&product_type_id=1
[8/55] Scraping: https://www.naiin.com/category?category_1_code=10&product_type_id=1
[9/55] Scraping: https://www.naiin.com/category?category_1_code=11&product_type_id=1
[10/55] Scraping: https://www.naiin.com/category?category_1_code=12&product_type_id=1
[11/55] Scraping: https://www.naiin.com/category?category_1_code=13&product_type_id=1
[12

In [4]:
cat_lv1_df

,product_type_id,category_1_code,category_1_title,status_code,url
0,1,1,หนังสือพระราชนิพนธ์,200,https://www.naiin.com/category?category_1_code...
1,1,2,นิยาย,200,https://www.naiin.com/category?category_1_code...
2,1,3,หนังสือวาย ยูริ,200,https://www.naiin.com/category?category_1_code...
3,1,4,หนังสือท่องเที่ยว,200,https://www.naiin.com/category?category_1_code...
4,1,5,บ้านและสวน,200,https://www.naiin.com/category?category_1_code...
5,1,8,งานอดิเรก งานฝีมือ,200,https://www.naiin.com/category?category_1_code...
6,1,9,สุขภาพ ความงาม,200,https://www.naiin.com/category?category_1_code...
7,1,10,แม่และเด็ก,200,https://www.naiin.com/category?category_1_code...
8,1,11,ธรรมะ ศาสนา และปรัชญา,200,https://www.naiin.com/category?category_1_code...
9,1,12,โหราศาสตร์ ดูดวง ฮวงจุ้ย,200,https://www.naiin.com/category?category_1_code...


In [5]:
import os
import pandas as pd

# 1. Get the current directory
current_directory = os.getcwd()

# 2. ระบุปลายทางไปที่โฟลเดอร์ /data
output_path = os.path.join(current_directory, "..", "data", "prod_cat_lv1.csv")

# 3. บันทึกไฟล์ด้วย Path ที่สร้างขึ้น
os.makedirs(os.path.dirname(output_path), exist_ok=True)
cat_lv1_df.to_csv(output_path, index=False, encoding='utf-8-sig', mode = 'w')

print(f"✅ Saved product URLs to: {output_path}")

✅ Saved product URLs to: d:\Project\book-scrap\ingestion\..\data\prod_cat_lv1.csv


## Category level 2

In [6]:
def get_category_lv2_title(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'th-TH,th;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.naiin.com/',
    }

    parsed_url = urlparse(url)
    params = parse_qs(parsed_url.query)
    p_id = params.get('product_type_id', [''])[0]
    cat1_code = params.get('category_1_code', [''])[0]
    cat2_code = params.get('categoryLv2Code', [''])[0]

    data = {
        'product_type_id': p_id,
        'category_1_code': cat1_code,
        'category_2_code': cat2_code,
        'category_2_title': "-",
        'status_code': None,
        'url': url
    }

    try:
        response = session.get(url, headers=headers, timeout=15)
        data['status_code'] = response.status_code

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            h1_tag = soup.find('h1', class_='tw-text-center')
            if h1_tag:
                data['category_2_title'] = h1_tag.get_text(strip=True)
        else:
            print(f"  [!] Status {response.status_code} for {url}")

    except Exception as e:
        print(f"  [!] Error at {url}: {e}")
        data['status_code'] = "Error"

    return data

def get_product_level_2():
    sitemap_url = "https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_CategoryLv2.xml"
    print("Fetching Category Level 2 Sitemap...")

    try:
        response = requests.get(sitemap_url)
        root = etree.fromstring(response.content)
        namespaces = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        all_urls = root.xpath('//ns:loc/text()', namespaces=namespaces)
    except Exception as e:
        print(f"Failed to read sitemap: {e}")
        return None

    target_urls = [u for u in all_urls]
    print(f"Total target URLs: {len(target_urls)}")

    results = []

    with requests.Session() as session:
        for i, url in enumerate(target_urls):
            print(f"[{i+1}/{len(target_urls)}] Scraping: {url}")

            data = get_category_lv2_title(url, session)
            results.append(data)

            time.sleep(1) 

    df = pd.DataFrame(results)

    return df

In [7]:
cat_lv2_df = get_product_level_2()

Fetching Category Level 2 Sitemap...
Total target URLs: 190
[1/190] Scraping: https://www.naiin.com/category?category_1_code=1&categoryLv2Code=54&product_type_id=1
[2/190] Scraping: https://www.naiin.com/category?category_1_code=1&categoryLv2Code=135&product_type_id=1
[3/190] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=6&product_type_id=1
[4/190] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=7&product_type_id=1
[5/190] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=8&product_type_id=1
[6/190] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=86&product_type_id=1
[7/190] Scraping: https://www.naiin.com/category?category_1_code=2&categoryLv2Code=134&product_type_id=1
[8/190] Scraping: https://www.naiin.com/category?category_1_code=3&categoryLv2Code=95&product_type_id=1
[9/190] Scraping: https://www.naiin.com/category?category_1_code=3&categoryLv2Code=140&product_type_id=1
[10/

In [8]:
import os
import pandas as pd

# 1. get the current directory
current_directory = os.getcwd()

# 2. ระบุปลายทางไปที่โฟลเดอร์ /data
output_path = os.path.join(current_directory , "..", "data", "prod_cat_lv2.csv")

# 3. บันทึกไฟล์ด้วย Path ที่สร้างขึ้น
os.makedirs(os.path.dirname(output_path), exist_ok=True)
cat_lv2_df.to_csv(output_path, index=False, encoding='utf-8-sig', mode = 'w')

print(f"✅ Saved product URLs to: {output_path}")

✅ Saved product URLs to: d:\Project\book-scrap\ingestion\..\data\prod_cat_lv2.csv


# Create product type id

In [2]:
import pandas as pd
import os

In [3]:
# manual check for product type_id
product_type_id_kv = {
    "product_type_id": [1, 2, 3, 4, 8],
    "product_type_description": ["หนังสือ", "นิตยสาร", "อีบุ๊ก", "อีแม็กกาซีน", "ไลฟ์สไตล์แอนด์กิ๊ฟ"]
}

In [4]:
product_type_df = pd.DataFrame(product_type_id_kv)
product_type_df

,product_type_id,product_type_description
0,1,หนังสือ
1,2,นิตยสาร
2,3,อีบุ๊ก
3,4,อีแม็กกาซีน
4,8,ไลฟ์สไตล์แอนด์กิ๊ฟ


In [7]:
# 1. get the current directory
current_directory = os.getcwd()

# 2. ระบุปลายทางไปที่โฟลเดอร์ /data
output_path = os.path.join(current_directory , "..", "data", "product_type.csv")

# 3. บันทึกไฟล์ด้วย Path ที่สร้างขึ้น
os.makedirs(os.path.dirname(output_path), exist_ok=True)
product_type_df.to_csv(output_path, index=False, encoding='utf-8-sig')

# Get product details url for scraping

In [3]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import datetime

In [4]:
SITEMAP_INDEX_URLS = [
    f"https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_ProductDetail-{i}.xml"
    for i in range(1, 6)
]

In [5]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'th-TH,th;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.naiin.com/',
}

In [6]:
def get_book_urls(url):
    print(f"Opening sub-sitemap: {url}")
    try:
        # ใช้ lxml-xml เพื่อความเร็วในการ parse XML ขนาดใหญ่
        response = requests.get(url, headers=HEADERS, timeout=30)
        soup = BeautifulSoup(response.content, 'lxml-xml')

        all_book_urls = [loc.text for loc in soup.find_all("loc")]
        print(f"Found {len(all_book_urls)} total books.")
        return all_book_urls
    except Exception as e:
        print(f"Error accessing sitemap {url}: {e}")
        return []

In [7]:
def get_all_book_url(SITEMAP_INDEX_URLS):
    all_results = []
    for sitemap_url in SITEMAP_INDEX_URLS:
        urls = get_book_urls(sitemap_url)
        
        for u in urls:
            # 1. ตัด / ที่ปิดท้ายออกก่อน แล้วค่อย split
            # ตัวอย่าง: ".../detail/123/" -> ".../detail/123" -> "123"
            p_id = u.strip('/').split('/')[-1]
            
            # 2. ป้องกันกรณีที่ url อาจจะสั้นเกินไปจนไม่ใช่ id
            if p_id:
                all_results.append({
                    'product_id': p_id,
                    'url': u,
                    'status': 'pending',
                    'discovered_at': datetime.datetime.now()
                })

    df = pd.DataFrame(all_results)
    
    # ตรวจสอบเบื้องต้น: ถ้ายังมี null ให้ drop ทิ้งไปเลย
    df = df.dropna(subset=['product_id'])
    
    return df

In [8]:
products_url = get_all_book_url(SITEMAP_INDEX_URLS)

Opening sub-sitemap: https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_ProductDetail-1.xml
Found 40000 total books.
Opening sub-sitemap: https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_ProductDetail-2.xml
Found 40000 total books.
Opening sub-sitemap: https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_ProductDetail-3.xml
Found 40000 total books.
Opening sub-sitemap: https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_ProductDetail-4.xml
Found 40000 total books.
Opening sub-sitemap: https://storage.naiin.com/system/application/bookstore/resource/sitemap/sitemap_ProductDetail-5.xml
Found 26966 total books.


In [9]:
products_url.count()

product_id       186966
url              186966
status           186966
discovered_at    186966
dtype: int64

In [ ]:
import os
import pandas as pd

# 1. กำหนดฐานของโปรเจกต์ (ถอยกลับไป 1 step จาก /ingestion)
base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# 2. ระบุปลายทางไปที่โฟลเดอร์ /data
output_path = os.path.join(base_dir, "data", "prod_urls.csv")

# 3. บันทึกไฟล์ด้วย Path ที่สร้างขึ้น
os.makedirs(os.path.dirname(output_path), exist_ok=True)
products_url.to_csv(output_path, index=False)

print(f"✅ Saved product URLs to: {output_path}")